In [41]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [80]:
df_lifespan = pd.read_csv("47180/cleaned_plant_lifespan_longevity.csv")
df_height_min = pd.read_csv("47176/cleaned_plant_height.csv")
df_height_max = pd.read_csv("47178/cleaned_plant_height_vegetative.csv")
df_leaf_area = pd.read_csv("leaf_area/leaf_area_cleaned.csv")
df_ssd = pd.read_csv("ssd/ssd_cleaned.csv")
bien_height = pd.read_csv("bien_traits_summary/bien_summary_plant_height.csv")
bien_leaf_area = pd.read_csv("bien_traits_summary/bien_summary_leaf_area.csv")
bien_sexual_system = pd.read_csv("bien_traits_summary/bien_summary_sexual_system.csv")
bien_swd = pd.read_csv("bien_traits_summary/bien_summary_stem_wood_density.csv")
df_gift_leaf_size = pd.read_csv("GIFT-db_traits/GIFT-db_leaf_size.csv")
df_gift_seed_length = pd.read_csv("GIFT-db_traits/GIFT-db_seed_length.csv")
df_gift_height = pd.read_csv("GIFT-db_traits/GIFT-db_plant_height.csv")
df_gift_ssd = pd.read_csv("GIFT-db_traits/GIFT-db_ssd.csv")
df_gift_lifestyle = pd.read_csv("GIFT-db_traits/GIFT-db_lifecycle.csv")
df_gift_sexual_reproduction = pd.read_csv("GIFT-db_traits/GIFT-db_reproduction_sexual.csv")
TR8_db_df_height = pd.read_csv("TR8_db_traits/TR8_db_df_height.csv")
TR8_db_df_lifespan = pd.read_csv("TR8_db_traits/TR8_db_lifespan.csv")
df_plant_genome = pd.read_csv("plant_genome/plant_genome.csv")

In [72]:
# helper function to standarize
def _std_species(df, col):
    out = df.copy()
    out["species"] = out[col].astype(str).str.strip().str.lower()
    return out.drop(columns=[col])

def _to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _stack_quant(df, trait, col_map):
    keep = ["species"]
    out = df[keep].copy()
    for stat, c in col_map.items():
        if c in df.columns:
            out[f"{trait}_{stat}"] = df[c]
    return out

def _mode(series):
    vc = series.dropna().value_counts()
    return vc.index[0] if not vc.empty else np.nan

In [81]:
# standarize species name
hm = _std_species(df_height_min, "AccSpeciesName")
hx = _std_species(df_height_max, "AccSpeciesName")
life = _std_species(df_lifespan, "AccSpeciesName")
la = _std_species(df_leaf_area, "AccSpeciesName")
ssd = _std_species(df_ssd, "AccSpeciesName")
pg = _std_species(df_plant_genome, "scientific_name")

bh = _std_species(bien_height, "scrubbed_species_binomial")
bla = _std_species(bien_leaf_area, "scrubbed_species_binomial")
bswd = _std_species(bien_swd, "scrubbed_species_binomial")

gls = _std_species(df_gift_leaf_size, "work_species")
gsl = _std_species(df_gift_seed_length, "work_species")
gh = _std_species(df_gift_height, "work_species")
gssd = _std_species(df_gift_ssd, "work_species")
glife = _std_species(df_gift_lifestyle, "work_species")

th = _std_species(TR8_db_df_height, "scientific_name")
tl = _std_species(TR8_db_df_lifespan, "scientific_name")

In [78]:
# plant height
plant_height_parts = [
    _stack_quant(gh, "plant_height", {
        "mean": "trait_value_Plant_height_mean",
        "min":  "trait_value_Plant_height_min",
        "max":  "trait_value_Plant_height_max",
    }),
    _stack_quant(bh, "plant_height", {
        "mean": "plant_height_m_mean",
        "min":  "plant_height_m_min",
        "max":  "plant_height_m_max",
    }),

    _stack_quant(hm, "plant_height", {"min": "min_height_min_m"}),
    _stack_quant(hx, "plant_height", {"max": "max_height_max_m"}),

    _stack_quant(th, "plant_height", {
        "mean": "plant_height_m_mean",
        "min":  "plant_height_m_min",
        "max":  "plant_height_m_max",
    }),
]


plant_height_all = pd.concat(plant_height_parts, ignore_index=True)

plant_height_summary = plant_height_all.groupby("species", as_index=False).agg(
    plant_height_mean=("plant_height_mean", "mean"),
    plant_height_min=("plant_height_min", "min"),
    plant_height_max=("plant_height_max", "max"),
)

plant_height_summary

,species,plant_height_mean,plant_height_min,plant_height_max
0,adansonia grandidieri,20.000000,NaN,NaN
1,aegle marmelos,10.000000,NaN,10.0000
2,aesculus hippocastanum,21.983333,15.0,39.0000
3,angophora floribunda,18.000000,15.0,30.0000
4,aquilaria malaccensis,20.000000,NaN,36.0000
...,...,...,...,...
508,voacanga thouarsii,12.500000,2.0,17.0000
509,xylocarpus granatum,10.000000,4.0,20.0000
510,zanthoxylum armatum,4.000000,NaN,NaN
511,ziziphus jujuba,11.477000,3.0,14.0208


In [79]:
plant_height_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 513 entries, 0 to 512
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            513 non-null    object 
 1   plant_height_mean  392 non-null    float64
 2   plant_height_min   329 non-null    float64
 3   plant_height_max   465 non-null    float64
dtypes: float64(3), object(1)
memory usage: 16.2+ KB


In [68]:
# leaf area

# need to standardize value
# la in g_mm2
# bla in m2

,leaf_area_m2_mean,leaf_area_m2_min,leaf_area_m2_max,unit,project_pi,species
0,6528.700000,6331.60,6725.8,mm2,K. Thompson; Price CA,aesculus hippocastanum
1,141.960000,108.10,180.3,mm2,Price CA; K. Thompson,buxus sempervirens
2,732.809091,165.60,2650.1,mm2,Milla R,capsicum annuum
3,1059.500000,1059.50,1059.5,mm2,D.K. Kunzmann,caragana arborescens
4,7035.000000,7035.00,7035.0,mm2,K. Thompson,carex acutiformis
...,...,...,...,...,...,...
131,188.000000,188.00,188.0,mm2,K. Thompson; Marx HE,veronica serpyllifolia
132,1799.540000,1435.80,2313.0,mm2,K. Thompson; de la Riva EG; Price CA,viburnum tinus
133,664.981420,186.25,1393.0,mm2,D.K. Kunzmann; K. Thompson; Marx HE,vicia sativa
134,1904.606667,1059.00,2413.0,mm2,K. Thompson; D.K. Kunzmann,vicia tenuifolia


In [48]:
# SWD and SSD

# should i merge them together?

In [49]:
# Lifespan
lifespan_parts = [
    _stack_quant(life, "life_exp", {"mean":"mean_life_exp"}),       
]

lifespan_summary = lifespan_all.groupby("species", as_index=False).agg(
    life_exp_mean=("life_exp_mean", "mean")
)

lifespan_summary

,species,life_exp_mean
0,aesculus hippocastanum,196.428571
1,angophora floribunda,100.000000
2,buxus sempervirens,550.000000
3,calotropis procera,4.000000
4,camellia japonica,27.000000
5,capsicum annuum,1.000000
6,caragana arborescens,33.000000
7,carex distans,6.140000
8,carya illinoinensis,300.000000
9,castanea dentata,300.000000


In [50]:
# Seed Length
seed_length_all = _stack_quant(gsl, "seed_length", {
    "mean":"trait_value_Seed_length_mean",
    "min":"trait_value_Seed_length_min",
    "max":"trait_value_Seed_length_max",
})
seed_length_summary = seed_length_all.groupby("species", as_index=False).agg(
    seed_length_mean=("seed_length_mean", "mean"),
    seed_length_min=("seed_length_min", "min"),
    seed_length_max=("seed_length_max", "max"),
)

seed_length_summary

,species,seed_length_mean,seed_length_min,seed_length_max
0,buxus sempervirens,5.0,5.0,5.0
1,camellia japonica,2.3,2.3,2.3
2,capsicum annuum,3.0,3.0,3.0
3,carex echinata,2.0,2.0,2.0
4,casuarina cunninghamiana,5.0,5.0,5.0
...,...,...,...,...
61,trifolium subterraneum,2.0,2.0,2.0
62,veronica scutellata,1.6,1.6,1.6
63,veronica serpyllifolia,1.0,NaN,NaN
64,vicia sativa,4.0,2.5,4.0


In [51]:
# Leaf size
leaf_size_all = _stack_quant(gls, "leaf_size", {"mean":"trait_value_Leaf_size_mean"})
leaf_size_summary = leaf_size_all.groupby("species", as_index=False).agg(
    leaf_size_mean=("leaf_size_mean", "mean")
)

leaf_size_summary

,species,leaf_size_mean
0,aesculus hippocastanum,65.28700
1,buxus sempervirens,1.38750
2,caragana arborescens,10.59500
3,carex acutiformis,70.35000
4,carex distans,2.65250
...,...,...
78,veronica serpyllifolia,1.88000
79,viburnum tinus,16.72950
80,vicia sativa,7.48850
81,vicia tenuifolia,19.04607


In [54]:
# Sexual system / sexual reproduction
sex_bien = _std_species(bien_sexual_system, "scrubbed_species_binomial")[["species", "sexual_system"]].copy()
sex_gift = _std_species(df_gift_sexual_reproduction, "work_species")[["species", "trait_value_Reproduction_sexual_1"]].copy()

sex_bien = sex_bien.rename(columns={"sexual_system":"sexual_reproduction"})
sex_gift = sex_gift.rename(columns={"trait_value_Reproduction_sexual_1":"sexual_reproduction"})

sex_all = pd.concat([sex_bien, sex_gift], ignore_index=True)
sexual_reproduction_summary = sex_all.groupby("species", as_index=False).agg(
    sexual_reproduction=("sexual_reproduction", _mode)
)

sexual_reproduction_summary

,species,sexual_reproduction
0,bruguiera gymnorhiza,Hermaphrodite
1,buxus sempervirens,monoecious
2,calotropis procera,Hermaphrodite
3,carex echinata,monoecious
4,carex extensa,monoecious
...,...,...
98,veronica scutellata,bisexual
99,veronica serpyllifolia,bisexual
100,vicia sativa,bisexual
101,viola palustris,bisexual


In [76]:
# Lifecycle
life_cat = glife[["species", "trait_value_Lifecycle_1"]].copy()
tl_cat = tl[["species", "LEDA_Lifespan_clean"]].copy()
life_cat = life_cat.rename(columns={"trait_value_Lifecycle_1":"lifecycle"})
tl_cat = tl_cat.rename(columns={"LEDA_Lifespan_clean": "lifecycle"})

lifecycle_all = pd.concat([life_cat, tl_cat], ignore_index=True)

lifecycle_summary = lifecycle_all.groupby("species", as_index=False).agg(
    lifecycle=("lifecycle", _mode)
)

lifecycle_summary

,species,lifecycle
0,adansonia grandidieri,perennial
1,aegilops bicornis,annual
2,aegilops sharonensis,annual
3,aegle marmelos,perennial
4,aesculus hippocastanum,perennial
...,...,...
443,xylocarpus granatum,perennial
444,zanthoxylum armatum,perennial
445,zizania palustris,annual
446,ziziphus jujuba,perennial


In [56]:
df_gift_ssd.columns

Index(['work_species', 'trait_value_SSD_mean', 'work_author'], dtype='object')

In [57]:
df_ssd_std = (
    df_ssd
    .rename(columns={
        "AccSpeciesName": "species",
        "ssd_mean_g_cm3": "ssd_mean"
    })
    [["species", "ssd_mean"]]
)
df_ssd_std["source"] = "SSD_db"

df_gift_ssd_std = (
    df_gift_ssd
    .rename(columns={
        "work_species": "species",
        "trait_value_SSD_mean": "ssd_mean"
    })
    [["species", "ssd_mean"]]
)
df_gift_ssd_std["source"] = "GIFT"

df_ssd_all = pd.concat(
    [df_ssd_std, df_gift_ssd_std],
    ignore_index=True
)
df_ssd_all

,species,ssd_mean,source
0,aegle marmelos,0.825728,SSD_db
1,aesculus hippocastanum,0.586531,SSD_db
2,angophora floribunda,0.725120,SSD_db
3,aquilaria malaccensis,0.320000,SSD_db
4,aquilaria sinensis,0.366158,SSD_db
...,...,...,...
202,liquidambar gracilipes,692.000000,GIFT
203,liquidambar chinensis,727.000000,GIFT
204,erythroxylum havanense,990.000000,GIFT
205,ormosia semicastrata,772.000000,GIFT


In [58]:
# compare ssd and wood density
df_ssd
bien_swd
df_gift_ssd

,work_species,trait_value_SSD_mean,work_author
0,aesculus hippocastanum,510.0,L.
1,buxus sempervirens,910.0,L.
2,castanea sativa,500.0,Mill.
3,casuarina cunninghamiana,803.0,Miq.
4,casuarina equisetifolia,859.0,L.
...,...,...,...
65,liquidambar gracilipes,692.0,(Hemsl.) Ickert-Bond & J.Wen
66,liquidambar chinensis,727.0,Champ. ex Benth.
67,erythroxylum havanense,990.0,Jacq.
68,ormosia semicastrata,772.0,Hance


In [62]:
df_merged = df_gift_ssd.merge(bien_swd, how="inner", left_on="work_species", right_on="scrubbed_species_binomial")
df_merged = df_merged.merge(df_ssd, how="inner", left_on="work_species", right_on="AccSpeciesName")
df_merged["trait_value_SSD_mean"] = df_merged["trait_value_SSD_mean"]/1000
df_ssd_and_wood_density = df_merged[["work_species", "trait_value_SSD_mean", "stem_wood_density_g_cm3_mean", "ssd_mean_g_cm3"]]

In [63]:
df_ssd_and_wood_density.to_csv("ssd_and_wood_density.csv", index=False)

In [83]:
# plant genome
plant_genome_summary = pg
plant_genome_summary

,redlistCategory,genome_size,species
0,Least Concern,1205385000,brasenia schreberi
1,Least Concern,290406122,bruguiera cylindrica
2,Least Concern,290406122,bruguiera gymnorhiza
3,Least Concern,290406122,bruguiera parviflora
4,Least Concern,290406122,bruguiera sexangula
...,...,...,...
667,Vulnerable,4110044999,sarracenia leucophylla
668,Vulnerable,1393650000,sideroxylon spinosum
669,Vulnerable,454770000,trichilia minutiflora
670,Vulnerable,1393650000,vitellaria paradoxa


In [84]:
# Meta Trait dataset

from functools import reduce

dfs = [
    plant_height_summary,
    lifespan_summary,
    seed_length_summary,
    leaf_size_summary,
    sexual_reproduction_summary,
    lifecycle_summary,
    plant_genome_summary,
]

final_traits = reduce(
    lambda left, right: left.merge(right, on="species", how="outer"),
    dfs
)
final_traits

,species,plant_height_mean,plant_height_min,plant_height_max,life_exp_mean,seed_length_mean,seed_length_min,seed_length_max,leaf_size_mean,sexual_reproduction,lifecycle,redlistCategory,genome_size
0,adansonia grandidieri,20.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,perennial,Endangered,1.432770e+09
1,aegle marmelos,10.000000,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,perennial,Near Threatened,4.890000e+08
2,aesculus hippocastanum,21.983333,15.0,39.0,196.428571,NaN,NaN,NaN,65.287,NaN,perennial,Vulnerable,6.259200e+08
3,angophora floribunda,18.000000,15.0,30.0,100.000000,NaN,NaN,NaN,NaN,NaN,perennial,Near Threatened,5.012250e+08
4,aquilaria malaccensis,20.000000,NaN,36.0,NaN,NaN,NaN,NaN,NaN,NaN,perennial,Critically Endangered,8.948700e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
700,metrosideros polymorpha var. glaberrima,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vulnerable,7.882069e+08
701,momordica enneaphylla,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vulnerable,1.376535e+09
702,populus ilicifolia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vulnerable,5.012250e+08
703,pterocarya macroptera,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vulnerable,9.914475e+08


In [86]:
final_traits = final_traits[
    final_traits["species"].isin(pg["species"])
]

In [87]:
final_traits.info()

<class 'pandas.core.frame.DataFrame'>
Index: 672 entries, 0 to 704
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   species              672 non-null    object 
 1   plant_height_mean    392 non-null    float64
 2   plant_height_min     325 non-null    float64
 3   plant_height_max     432 non-null    float64
 4   life_exp_mean        50 non-null     float64
 5   seed_length_mean     41 non-null     float64
 6   seed_length_min      57 non-null     float64
 7   seed_length_max      60 non-null     float64
 8   leaf_size_mean       83 non-null     float64
 9   sexual_reproduction  103 non-null    object 
 10  lifecycle            448 non-null    object 
 11  redlistCategory      672 non-null    object 
 12  genome_size          672 non-null    float64
dtypes: float64(9), object(4)
memory usage: 73.5+ KB


In [88]:
final_traits.to_csv("final_traits.csv", index=False)